# Day 3 (Week5): Full RAG Implementation

In [12]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_openai import ChatOpenAI # OpenAI LLM model absttract method (.invoke())
from langchain_ollama import ChatOllama # Ollama LLM model absttract method (.invoke())
from langchain_anthropic import ChatAnthropic # Anthropic LLM model absttract method (.invoke())

from langchain_chroma import Chroma
import gradio as gr
from langchain_core.messages import SystemMessage, HumanMessage

In [19]:
MODEL = "gpt-5-mini"
db_name = "vector_db"
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key:
    print("Key is loaded!!")
else:
    print("Key not loaded!")

Key is loaded!!


## Connecting to Chroma

In [14]:
embedding_model = "all-MiniLM-L6-v2" # HuggingFace openSource model

embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
vectorstore = Chroma(persist_directory = db_name, embedding_function= embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [20]:
retriver = vectorstore.as_retriever()
llm = ChatOpenAI(temperature = 0, model_name=MODEL)

### Testing the retriver & LLM's invoke method

In [21]:
retriver.invoke("Who is Avery?")

[Document(id='f67d98c3-8a90-49cc-8e65-32b0db023eff', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [ ]:
# Rigth now the model wouldn't answer correctly since we are not feeding it relevant data from the vector store.
llm.invoke("Who is Avery?")

AIMessage(content='I don’t have enough context — “Avery” can refer to many people, characters, or brands. Do you mean a person, a fictional character, or something else (a company/brand)?\n\nSome common possibilities:\n- Avery as a given name/surname (gender‑neutral).  \n- Avery Brooks — actor best known as Captain Benjamin Sisko on Star Trek: Deep Space Nine.  \n- Avery Johnson — former NBA player and coach.  \n- Avery Bradley — NBA guard.  \n- Avery Fisher — audio equipment philanthropist (Avery Fisher Prize).  \n- Avery Dennison (often just “Avery”) — company that makes labels, stickers, and office/packaging products.  \n- Avery — a minor Death Eater character in the Harry Potter books.\n\nIf you tell me where you saw “Avery” (book, show, news article, workplace, etc.) or give a last name, I’ll identify the right one.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 777, 'prompt_tokens': 10, 'total_tokens': 787, 'completion_tokens_detail

## Putting them together!!

In [23]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [26]:
def question_answer(question, history):
    docs = retriver.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context = context)
    response = llm.invoke([SystemMessage(content = system_prompt), HumanMessage(content = question)])
    return response.content

In [ ]:
print(question_answer("Who is Alex Chem?",[]))

Do you mean Alex Chen or Alex Thomson? 

- Alex Chen: According to our records, Alex Chen is a valued member of Insurellm who contributes to innovative backend solutions that help shape the future of our insurance technology.

- Alex Thomson: Alex Thomson is a Sales Development Representative (SDR) based in Austin, Texas (born March 15, 1995). Alex joined Insurellm in November 2022, was promoted to Team Lead for special projects in January 2023, developed onboarding training for new SDRs in August 2023, and currently leads a team of five SDRs while collaborating with marketing. Alex is active on the Diversity & Inclusion committee and has led team-building and outreach efforts.

Tell me which name you meant and I can give more details.


## Creating a gradio interface for question_answer()

In [ ]:
gr.ChatInterface(question_answer).launch()

c:\Users\yaduk\Desktop\AI_Study\study_section\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
